In [8]:
import numpy
import time

import matplotlib.pyplot as plt
import os
import ismrmrd

from sirf.Gadgetron import AcquisitionData

In [9]:
fname = 'work/data/h5/1meas_MID00619_FID129157_AI_RECON_SEQD_512_GF4_AX_RL.h5'
#fname = 'work/data/h5/1meas_MID00614_FID129152_CONVENTIONAL_RECON_SEQD_GF2_AX_RL.h5'

In [10]:
acq_data = AcquisitionData(fname)

reading from work/data/h5/1meas_MID00619_FID129157_AI_RECON_SEQD_512_GF4_AX_RL.h5 using ignore mask 0000 0000 0000 0000 0000 0000 0000 0000 0000 0000 0000 0000 0000 0000 0000 0000 

Started reading acquisitions from work/data/h5/1meas_MID00619_FID129157_AI_RECON_SEQD_512_GF4_AX_RL.h5
0%..10%..20%..30%..40%..50%..60%..70%..80%..90%..100%..
Finished reading acquisitions from work/data/h5/1meas_MID00619_FID129157_AI_RECON_SEQD_512_GF4_AX_RL.h5


In [11]:
print(acq_data.get_header())

<?xml version="1.0"?>
<ismrmrdHeader xmlns="http://www.ismrm.org/ISMRMRD" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance" xmlns:xs="http://www.w3.org/2001/XMLSchema" xsi:schemaLocation="http://www.ismrm.org/ISMRMRD ismrmrd.xsd">
	<studyInformation>
		<studyTime>19:52:23</studyTime>
	</studyInformation>
	<measurementInformation>
		<measurementID>183334_30000026030606544496500000018_30000026030606544496500000018_619</measurementID>
		<patientPosition>FFS</patientPosition>
		<protocolName>BOOST SEQD 512 UFLEXL no spine GF2 AX RL pd_tse_fs_sag</protocolName>
		<measurementDependency>
			<dependencyType>SenMap</dependencyType>
			<measurementID>183334_30000026030606544496500000018_30000026030606544496500000018_607</measurementID>
		</measurementDependency>
		<measurementDependency>
			<dependencyType>Noise</dependencyType>
			<measurementID>183334_30000026030606544496500000018_30000026030606544496500000018_607</measurementID>
		</measurementDependency>
		<frameOfReferenceUID>1.3.12.2.

In [12]:
def change_ismrmrd(full_filename_in, full_filename_out):
    if full_filename_in == full_filename_out:
        raise ValueError('Input and output filename are the same. This would overwrite the original data.')

    # Set trajectory and save
    if os.path.exists(full_filename_out) == 1:
        os.remove(full_filename_out)
        print('{} deleted'.format(full_filename_out))
        
    with ismrmrd.File(full_filename_in, 'r') as file:
        ds = file[list(file.keys())[0]]
        ismrmrd_header = ds.header
        acquisitions = ds.acquisitions[:]

    # Modify header
    ismrmrd_header.encoding[0].encodedSpace.matrixSize.y = 512
    
    # Create new file
    ds = ismrmrd.Dataset(full_filename_out)
    ds.write_xml_header(ismrmrd_header.toXML())

    for acq in acquisitions:
        ds.append_acquisition(acq)
    ds.close()
    

In [13]:
change_ismrmrd(fname, fname.replace('.h5', '_mod.h5'))